# Prepare Forest Sound Dataset

เป้าหมาย: เตรียมข้อมูลจาก `forest_sound_dataset` ให้พร้อมเทรนโมเดล RakForest โดยแบ่งชุดข้อมูลก่อน แล้วแปลงเป็น **mono, 16 kHz, 5 วินาที**.

> ไฟล์เสียงที่สร้างขึ้นจะอยู่ใน `processed/` และถูก Git ignore อัตโนมัติเพราะเป็น `.wav`.


## 1. Import และตั้งค่า


In [1]:
from pathlib import Path
from collections import Counter
import re
import warnings

import librosa
import numpy as np
import pandas as pd
import soundfile as sf
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')

EXPECTED_CLASSES = ['fire', 'logging', 'natural sound', 'poaching']
AUDIO_EXTENSIONS = {'.wav', '.wave', '.aiff', '.aif', '.flac', '.mp3', '.m4a', '.ogg', '.opus', '.aac', '.wma'}
TARGET_SAMPLE_RATE = 16_000
CLIP_SECONDS = 5
TARGET_SAMPLES = TARGET_SAMPLE_RATE * CLIP_SECONDS
RANDOM_SEED = 42
MAX_SEGMENTS_PER_SOURCE = 6  # กันคลิปยาวมากครอบงำข้อมูล

def find_dataset_dir():
    candidates = [Path('forest_sound_dataset'), Path.cwd() / 'forest_sound_dataset',
                  Path.cwd().parent / 'forest_sound_dataset', Path.cwd().parent.parent / 'forest_sound_dataset']
    for candidate in candidates:
        if candidate.is_dir():
            return candidate.resolve()
    raise FileNotFoundError('ไม่พบโฟลเดอร์ forest_sound_dataset')

DATASET_DIR = find_dataset_dir()
PROJECT_ROOT = DATASET_DIR.parent
PROCESSED_DIR = PROJECT_ROOT / 'processed' / 'forest_sound_16k_5s'
MANIFEST_DIR = PROJECT_ROOT / 'manifests'
MANIFEST_DIR.mkdir(exist_ok=True)

print(f'Dataset: {DATASET_DIR}')
print(f'Processed output: {PROCESSED_DIR}')


Dataset: /Users/guidegiegg/งาน/โปรเจคเล็กๆ/PrePair_Sound/forest_sound_dataset
Processed output: /Users/guidegiegg/งาน/โปรเจคเล็กๆ/PrePair_Sound/processed/forest_sound_16k_5s


## 2. สร้างรายการไฟล์ที่อ่านได้

ไฟล์ที่ EDA ตรวจพบว่าอ่านไม่ได้จะถูกข้าม และมีรายงานแยกไว้.


In [2]:
def audio_is_readable(path: Path) -> tuple[bool, str]:
    try:
        try:
            sf.info(path)
        except Exception:
            librosa.load(path, sr=None, mono=True, duration=0.1)
        return True, ''
    except Exception as exc:
        return False, f'{type(exc).__name__}: {exc}'

rows, bad_rows = [], []
for label in EXPECTED_CLASSES:
    for path in sorted((DATASET_DIR / label).rglob('*')):
        if not (path.is_file() and path.suffix.lower() in AUDIO_EXTENSIONS):
            continue
        ok, error = audio_is_readable(path)
        record = {'label': label, 'source_path': str(path), 'relative_path': str(path.relative_to(DATASET_DIR))}
        if ok:
            rows.append(record)
        else:
            bad_rows.append({**record, 'error': error})

sources_df = pd.DataFrame(rows)
bad_files_df = pd.DataFrame(bad_rows)
print(f'ใช้ได้: {len(sources_df):,} ไฟล์ | ข้ามเพราะอ่านไม่ได้: {len(bad_files_df):,} ไฟล์')
display(sources_df['label'].value_counts().reindex(EXPECTED_CLASSES).rename('usable_files').to_frame())
if not bad_files_df.empty:
    display(bad_files_df[['relative_path', 'error']])


ใช้ได้: 2,761 ไฟล์ | ข้ามเพราะอ่านไม่ได้: 1 ไฟล์


,usable_files
label,
fire,335
logging,455
natural sound,1363
poaching,608


,relative_path,error
0,fire/755366__goochiano__wild-fire.m4a,LibsndfileError: Error opening '/Users/guidegi...


## 3. แบ่ง Train / Validation / Test ก่อนตัดเสียง

แบ่งต้นฉบับก่อนเพื่อไม่ให้ segment จากไฟล์เดียวกันหลุดไปอยู่คนละชุด ซึ่งจะทำให้คะแนนโมเดลดูสูงเกินจริง.


In [3]:
train_df, temp_df = train_test_split(
    sources_df, test_size=0.30, random_state=RANDOM_SEED, stratify=sources_df['label']
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=RANDOM_SEED, stratify=temp_df['label']
)

split_sources = pd.concat([
    train_df.assign(split='train'),
    val_df.assign(split='val'),
    test_df.assign(split='test'),
], ignore_index=True)

split_summary = pd.crosstab(split_sources['label'], split_sources['split']).reindex(EXPECTED_CLASSES)
split_summary['total'] = split_summary.sum(axis=1)
display(split_summary)
print(f"Total: train={len(train_df):,}, val={len(val_df):,}, test={len(test_df):,}")


split,test,train,val,total
label,,,,
fire,50,234,51,335
logging,69,318,68,455
natural sound,205,954,204,1363
poaching,91,426,91,608


Total: train=1,932, val=414, test=415


## 4. ฟังก์ชันแปลงเสียงเป็น 16 kHz / mono / 5 วินาที

- คลิปสั้น: เติม silence ด้านท้าย
- คลิปยาว: ตัดเป็นช่วง 5 วินาทีแบบไม่ทับกัน แต่จำกัดสูงสุด 6 ช่วงต่อไฟล์


In [4]:
def make_segments(y: np.ndarray) -> list[np.ndarray]:
    if len(y) <= TARGET_SAMPLES:
        return [np.pad(y, (0, TARGET_SAMPLES - len(y)))]

    starts = list(range(0, len(y) - TARGET_SAMPLES + 1, TARGET_SAMPLES))
    if len(starts) > MAX_SEGMENTS_PER_SOURCE:
        starts = np.linspace(0, starts[-1], MAX_SEGMENTS_PER_SOURCE, dtype=int).tolist()
    return [y[start:start + TARGET_SAMPLES] for start in starts]

def safe_stem(relative_path: str) -> str:
    stem = Path(relative_path).stem
    return re.sub(r'[^A-Za-z0-9_-]+', '_', stem)


## 5. สร้าง Dataset ที่ผ่าน preprocessing

Cell นี้จะสร้างไฟล์ใหม่จำนวนมากใน `processed/` และอาจใช้เวลาหลายนาที. รันเพียงครั้งแรก; ถ้าต้องการสร้างใหม่ ให้ลบ `processed/forest_sound_16k_5s` ก่อน.


In [5]:
processed_rows, failures = [], []

for index, row in split_sources.reset_index(drop=True).iterrows():
    source_path = Path(row['source_path'])
    try:
        y, _ = librosa.load(source_path, sr=TARGET_SAMPLE_RATE, mono=True)
        segments = make_segments(y)
        output_dir = PROCESSED_DIR / row['split'] / row['label']
        output_dir.mkdir(parents=True, exist_ok=True)

        for segment_index, segment in enumerate(segments):
            output_path = output_dir / f"{safe_stem(row['relative_path'])}__seg{segment_index:02d}.wav"
            sf.write(output_path, segment, TARGET_SAMPLE_RATE, subtype='PCM_16')
            processed_rows.append({
                'split': row['split'], 'label': row['label'], 'source_relative_path': row['relative_path'],
                'segment_index': segment_index, 'processed_path': str(output_path.relative_to(PROJECT_ROOT)),
                'sample_rate': TARGET_SAMPLE_RATE, 'duration_sec': CLIP_SECONDS,
            })
    except Exception as exc:
        failures.append({'relative_path': row['relative_path'], 'error': f'{type(exc).__name__}: {exc}'})

    if (index + 1) % 100 == 0 or index + 1 == len(split_sources):
        print(f'Processed {index + 1:,}/{len(split_sources):,} source files')

processed_df = pd.DataFrame(processed_rows)
failures_df = pd.DataFrame(failures)
print(f'สร้างไฟล์พร้อมเทรน: {len(processed_df):,} segments | ล้มเหลวระหว่างแปลง: {len(failures_df):,}')


Processed 100/2,761 source files
Processed 200/2,761 source files
Processed 300/2,761 source files
Processed 400/2,761 source files
Processed 500/2,761 source files
Processed 600/2,761 source files
Processed 700/2,761 source files
Processed 800/2,761 source files
Processed 900/2,761 source files
Processed 1,000/2,761 source files
Processed 1,100/2,761 source files
Processed 1,200/2,761 source files
Processed 1,300/2,761 source files
Processed 1,400/2,761 source files
Processed 1,500/2,761 source files
Processed 1,600/2,761 source files
Processed 1,700/2,761 source files
Processed 1,800/2,761 source files
Processed 1,900/2,761 source files
Processed 2,000/2,761 source files
Processed 2,100/2,761 source files
Processed 2,200/2,761 source files
Processed 2,300/2,761 source files
Processed 2,400/2,761 source files
Processed 2,500/2,761 source files
Processed 2,600/2,761 source files
Processed 2,700/2,761 source files
Processed 2,761/2,761 source files
สร้างไฟล์พร้อมเทรน: 3,856 segments | ล

## 6. ตรวจผลลัพธ์และบันทึก manifest


In [6]:
prepared_summary = pd.crosstab(processed_df['label'], processed_df['split']).reindex(EXPECTED_CLASSES, fill_value=0)
prepared_summary['total'] = prepared_summary.sum(axis=1)
display(prepared_summary)

split_sources.to_csv(MANIFEST_DIR / 'forest_sound_source_splits.csv', index=False)
processed_df.to_csv(MANIFEST_DIR / 'forest_sound_processed_manifest.csv', index=False)
bad_files_df.to_csv(MANIFEST_DIR / 'forest_sound_excluded_files.csv', index=False)
failures_df.to_csv(MANIFEST_DIR / 'forest_sound_processing_failures.csv', index=False)

print('บันทึก manifests ใน:', MANIFEST_DIR)
print('ขั้นถัดไป: สร้าง Mel-Spectrogram และเทรน CNN')


split,test,train,val,total
label,,,,
fire,220,989,221,1430
logging,69,318,68,455
natural sound,205,954,204,1363
poaching,91,426,91,608


บันทึก manifests ใน: /Users/guidegiegg/งาน/โปรเจคเล็กๆ/PrePair_Sound/manifests
ขั้นถัดไป: สร้าง Mel-Spectrogram และเทรน CNN
